# Clase 4 – Refinamiento de Datos y Tratamiento de Duplicados y Valores Erróneos
### Limpieza y preparación de datos

**Temas:** Permutación · `sort_values()` múltiple · `duplicated()` · `drop_duplicates()` · `replace()`

---

In [ ]:
import pandas as pd
import numpy as np
print(f'pandas {pd.__version__} cargado ✅')

---
## Slide 3 – Desafío Inicial: Base de Datos de Fidelización

Problemas detectados: clientes repetidos, mismos nombres con distinta región, filas desordenadas,
valores de contacto como `None` e inconsistencias como `'altaa'`.

**Archivo:** `clientes_registros.csv`

In [ ]:
# ── Slide 3: Carga del desafío inicial ───────────────────────────────────────

import pandas as pd
df = pd.read_csv('clientes_registros.csv')
print(f'Filas: {df.shape[0]} | Columnas: {list(df.columns)}')
print(df.head(8))
print(f'\nDuplicados completos: {df.duplicated().sum()}')
print(f'Duplicados por cliente_id: {df.duplicated(subset=["cliente_id"]).sum()}')
print(f'\nValores en frecuencia_contacto:')
print(df['frecuencia_contacto'].value_counts(dropna=False))

---
## Slides 5–6 – Permutación de la Data

In [ ]:
# ── Slide 6: Permutación — frac=1 reorganiza el 100% de las filas ─────────────
# reset_index(drop=True): reindexa y elimina el índice anterior

df.sample(frac=1).reset_index(drop=True)

In [ ]:
# ── Slide 6: Ejemplo aplicado — permutación reproducible con random_state ──────
# Garantiza resultados repetibles (útil en producción o pruebas)

clientes_permutado = df.sample(frac=1, random_state=42).reset_index(drop=True)

---
## Slide 7 – Ordenamiento por Una o Múltiples Columnas

In [ ]:
# ── Slide 7: sort_values() por una columna ────────────────────────────────────

df.sort_values(by='compras', ascending=False)

In [ ]:
# ── Slide 7: sort_values() por múltiples columnas ─────────────────────────────

df.sort_values(by=['region', 'nombre'], ascending=[True, False])

---
## Slide 11 – Detección y Eliminación de Duplicados

In [ ]:
# ── Slide 11: duplicated() — serie booleana (True = duplicado) ────────────────

df.duplicated()

In [ ]:
# ── Slide 11: drop_duplicates() — eliminar filas repetidas completas ──────────

df_sin_duplicados = df.drop_duplicates()

In [ ]:
# ── Slide 11: drop_duplicates(subset) — por columnas específicas ──────────────
# Elimina entradas donde cliente_id se repite, manteniendo la primera

df.drop_duplicates(subset=['cliente_id'])

---
## Slide 15 – Reemplazo de Valores

In [ ]:
# ── Slide 15: Reemplazar un valor específico ──────────────────────────────────

df['frecuencia_contacto'].replace('altaa', 'alta', inplace=True)

In [ ]:
# ── Slide 15: Reemplazar múltiples valores simultáneamente ───────────────────

df.replace({'altaa': 'alta', None: 'desconocido'}, inplace=True)

In [ ]:
# ── Slide 15: Reemplazar con condiciones (loc) ────────────────────────────────
# Corrige datos inválidos sin eliminar el registro

df.loc[df['compras'] == 0, 'compras'] = 1

---
## Slides 17–19 – Actividad Guiada: Limpieza Final de Registros de Clientes

**Objetivo (Slide 18):** Permutar, ordenar, eliminar duplicados y corregir valores.

**Archivo:** `clientes_registros.csv`

In [ ]:
# ── Slide 19: Cargar el archivo ───────────────────────────────────────────────

import pandas as pd
df = pd.read_csv('clientes_registros.csv')
print(f'Filas originales: {len(df)}')
print(df.head())

In [ ]:
# ── Slide 19, Paso 1: Permutar el orden de las filas ─────────────────────────

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print('DataFrame permutado:')
print(df.head())

In [ ]:
# ── Slide 19, Paso 2: Ordenar por 'compras' de mayor a menor ─────────────────

df = df.sort_values(by='compras', ascending=False)
print(df.head())

In [ ]:
# ── Slide 19, Paso 3: Eliminar duplicados por cliente_id ──────────────────────

df = df.drop_duplicates(subset=['cliente_id'])
print(f'Filas después de eliminar duplicados por cliente_id: {len(df)}')

In [ ]:
# ── Slide 19, Paso 4: Reemplazar nulos en 'frecuencia_contacto' ───────────────

df['frecuencia_contacto'] = df['frecuencia_contacto'].fillna('desconocido')

# ── Slide 19, Paso 5: Corregir todas las variantes de 'alta' ──────────────────
# Incluye: 'altaa', 'Altaa', 'ALTA', '  alta ' (con espacios)

df['frecuencia_contacto'] = df['frecuencia_contacto'].str.strip().str.lower()
df['frecuencia_contacto'] = df['frecuencia_contacto'].replace({
    'altaa': 'alta',
    'medía': 'media',
})

print('Valores únicos después de corregir:')
print(df['frecuencia_contacto'].value_counts(dropna=False))

In [ ]:
# ── Slide 19, Paso 6: Guardar resultado como clientes_limpios.csv ─────────────

df.to_csv('clientes_limpios.csv', index=False)
print(f'✅ clientes_limpios.csv exportado: {len(df)} registros')

In [ ]:
# ── Slide 19: Resumen del proceso de limpieza ─────────────────────────────────

df_orig = pd.read_csv('clientes_registros.csv')
print('=== RESUMEN DE LIMPIEZA ===')
print(f'Filas originales:             {len(df_orig)}')
print(f'Filas después de limpieza:    {len(df)}')
print(f'Registros eliminados:         {len(df_orig) - len(df)}')
print(f'Duplicados completos:         {df_orig.duplicated().sum()}')
print(f'Duplicados por cliente_id:    {df_orig.duplicated(subset=["cliente_id"]).sum()}')
print(f'Nulos en frecuencia_contacto: {df_orig["frecuencia_contacto"].isna().sum()}')

---
## Slides 21–23 – Actividad Autónoma: Depuración de Reclamos de Productos

**Contexto (Slide 22):** Archivo con duplicados y etiquetas inconsistentes (`'electrónicco'`, `'sin categ.'`).

---
**Completa las celdas marcadas con `# 📝 TU CÓDIGO AQUÍ`**

In [ ]:
# ── Slide 23: Crear el DataFrame de reclamos ──────────────────────────────────

import pandas as pd
df = pd.DataFrame({
    'reclamo_id': [11, 12, 13, 14, 15, 13],
    'producto': ['TV', 'Celular', 'Tablet', 'TV', 'Celular', 'Tablet'],
    'categoria_defecto': ['electrónico', 'electrónicco', 'electrónico',
                          'electrónico', 'sin categ.', 'electrónico']
})
print(df)

In [ ]:
# ── Slide 23, Paso 1: Eliminar duplicados por reclamo_id ──────────────────────

# 📝 TU CÓDIGO AQUÍ
# df = df.drop_duplicates(subset=[...])

In [ ]:
# ── Slide 23, Paso 2: Reemplazar 'electrónicco' → 'electrónico' ───────────────

# 📝 TU CÓDIGO AQUÍ
# df['categoria_defecto'].replace(..., inplace=True)

In [ ]:
# ── Slide 23, Paso 3: Reemplazar 'sin categ.' → 'desconocido' ────────────────

# 📝 TU CÓDIGO AQUÍ

In [ ]:
# ── Slide 23, Paso 4: Ordenar por 'producto' ─────────────────────────────────

# 📝 TU CÓDIGO AQUÍ
# df = df.sort_values(by=...)

In [ ]:
# ── Slide 23, Paso 5: Guardar como reclamos_limpios.csv ──────────────────────

# 📝 TU CÓDIGO AQUÍ
# df.to_csv('reclamos_limpios.csv', index=False)

### Preguntas de reflexión y cierre (Slide 20)

1. ¿Qué criterio usaste para decidir qué duplicado mantener (`keep='first'` o `keep='last'`)?
2. ¿Qué impacto tienen los errores de ortografía o valores vacíos en una campaña comercial?
3. ¿Qué diferencia hay entre eliminar un duplicado y reemplazar su valor?